# Product Category–Driven Profitability Classification  
## Demand Forecasting for Retail Inventory Optimization

### Objective

In the earlier phases of this project, the dataset was treated primarily as a structured retail demand forecasting dataset.  
However, for this notebook, the analysis is reframed from the **product-category and stocking decision perspective**.

The instructor's guidance is to answer questions such as:

- Which product types or categories are more profitable to sell?
- Which products should be prioritized for stocking?
- How can textual business descriptors such as category, brand, sub-category, and promotion type be used for classification?

This notebook therefore treats the problem as a **multi-class classification task** driven primarily by product-related textual and categorical descriptors.

---

## Modeling Goal

We create a business-oriented target called **`Stocking_Priority_Class`**, which represents whether a product record is a:

- **Low priority**
- **Medium priority**
- **High priority**

stocking decision from a profit-density perspective.

---

## Models Compared

We compare three modeling strategies:

1. **Logistic Regression** with TF-IDF features  
2. **A Custom Deep Learning Text Classifier**  
3. **A Hugging Face Pre-trained Transformer Classifier**  

---

## Why This Reformulation Makes Sense

Retailers do not only ask:

> "How many units will sell?"

They also ask:

- Which types of products should be stocked more aggressively?
- Which products give better value relative to space consumed?
- Which categories create stronger commercial returns?

That is why this notebook shifts from pure regression to **category-driven profitability classification**.

# Mathematical Formulation

## Step 1: Profit Per Unit

For each row, profit per unit is defined as:

$
	ext{Profit Per Unit} = 	ext{Current Price} - 	ext{Unit Cost}
$

---

## Step 2: Profit Per Record

The total profit represented by one record is:

$
	ext{Profit Per Record} = 	ext{Profit Per Unit} 	imes 	ext{Daily Units Sold}
$

---

## Step 3: Profit Density

Because stocking decisions are constrained by available shelf space, we define:

$
	ext{Profit Density} = rac{	ext{Profit Per Record}}{	ext{Shelf Capacity}}
$

This helps us compare products not just by sales, but by **profit relative to space consumed**.

---

## Step 4: Multi-Class Target

We convert profit density into a 3-class business decision target using quantile-based buckets:

$
	ext{Stocking Priority Class} \in \{ 	ext{Low}, 	ext{Medium}, 	ext{High} \}
$

This creates a balanced multi-class classification problem.

---

## Multi-Class Logistic Regression

For multiclass logistic regression, the probability of class $k$ is modeled using the softmax function:

$
P(y = k \mid x) = rac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}
$

where

$
z_k = w_k^T x + b_k
$

The model is trained by minimizing categorical cross-entropy:

$
\mathcal{L} = - \sum_{i=1}^{N} \sum_{k=1}^{K} y_{ik}\log(\hat{y}_{ik})
$

---

## Deep Learning and Transformer Models

The custom deep learning model and the Hugging Face transformer model are also trained for the same multi-class classification objective, using:

$
	ext{Softmax Output} ightarrow 	ext{Cross-Entropy Loss}
$

# Notebook Workflow

1. Load and inspect the updated retail dataset  
2. Construct a business-oriented multi-class target  
3. Prepare text-style input from product/category/business fields  
4. Train a **Logistic Regression + TF-IDF baseline**  
5. Train a **Custom Deep Learning classifier**  
6. Fine-tune a **Hugging Face pre-trained transformer**  
7. Compare all three models using:
   - Accuracy
   - Precision
   - Recall
   - F1-score
8. Summarize business interpretation

In [ ]:
# ============================================================
# Optional package installation cell for Google Colab
# ============================================================
# Uncomment and run only if the required packages are missing.
#
# !pip -q install transformers datasets accelerate evaluate sentencepiece

In [ ]:
# ============================================================
# Import required libraries
# ============================================================

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Hugging Face imports
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# Visualization settings
sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load Dataset

The updated dataset is loaded from the project files.

We will inspect:
- shape
- column names
- sample rows

In [ ]:
# Load dataset
df = pd.read_csv("master_retail_dataset_v2.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.head()

# Dataset Perspective for This Notebook

Although the dataset contains many numeric variables, the instructor's direction is to analyze the dataset from the **product category and stocking decision perspective**.

Therefore, the input side of the model will emphasize fields such as:

- `Category`
- `Product_Name`
- `Sub_Category`
- `Brand_Name`
- `Brand_Tier`
- `Promotion_Type`
- `Festival_Name`
- `Festival_Type`
- `Store_Type`

These are not long free-text documents, but they do capture useful **business semantics** about products and selling context.

# Create Business-Oriented Target

We now create:

- `Profit_Per_Unit`
- `Profit_Per_Record`
- `Profit_Density`
- `Stocking_Priority_Class`

This target is designed to reflect:

- profitability
- demand
- stocking space efficiency

In [ ]:
# ============================================================
# Create business-oriented target
# ============================================================

# Profit per unit sold
df["Profit_Per_Unit"] = df["Current_Price"] - df["Unit_Cost"]

# Total profit contribution of one record
df["Profit_Per_Record"] = df["Profit_Per_Unit"] * df["Daily_Units_Sold"]

# Profit density: profit relative to shelf capacity
df["Profit_Density"] = df["Profit_Per_Record"] / df["Shelf_Capacity"].replace(0, np.nan)

# Fill any unexpected missing values in profit density
df["Profit_Density"] = df["Profit_Density"].fillna(df["Profit_Density"].median())

# Create 3 balanced classes using quantiles
df["Stocking_Priority_Class"] = pd.qcut(
    df["Profit_Density"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Display target distribution
print(df["Stocking_Priority_Class"].value_counts())
df[["Profit_Per_Unit", "Profit_Per_Record", "Profit_Density", "Stocking_Priority_Class"]].head()

## Observation

The target classes are created using quantile-based buckets, which makes the classes approximately balanced.

This is useful because balanced classes reduce bias during multi-class model training and make evaluation more reliable.

# Select Text-Like Business Features

We combine the following fields into a single business text representation for each row:

- category
- product name
- sub-category
- brand
- brand tier
- promotion type
- festival name / type
- store type

This allows all three classifiers to work on the same input representation.

In [ ]:
# ============================================================
# Define text-like business columns
# ============================================================

text_columns = [
    "Category",
    "Product_Name",
    "Sub_Category",
    "Brand_Name",
    "Brand_Tier",
    "Promotion_Type",
    "Festival_Name",
    "Festival_Type",
    "Store_Type"
]

# Fill missing values defensively
for col in text_columns:
    df[col] = df[col].fillna("Unknown")

# Create one combined text field per row
df["combined_text"] = df[text_columns].astype(str).agg(" ".join, axis=1)

# Light cleanup
df["combined_text"] = (
    df["combined_text"]
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df[["combined_text", "Stocking_Priority_Class"]].head()

# Encode Labels

The target labels are converted from:

- Low
- Medium
- High

into numeric class IDs:

- 0
- 1
- 2

In [ ]:
# ============================================================
# Encode target labels
# ============================================================

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["Stocking_Priority_Class"])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", label_mapping)

df[["Stocking_Priority_Class", "label"]].head()

# Train / Validation / Test Split

We split the dataset into:

- **Training set**
- **Validation set**
- **Test set**

We use **stratified splitting** so that the class balance is preserved across all subsets.

In [ ]:
# ============================================================
# Stratified train/validation/test split
# ============================================================

X = df["combined_text"]
y = df["label"]

# First split: train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Second split: train vs validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.20,
    random_state=SEED,
    stratify=y_trainval
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

# Utility Functions for Evaluation

We define reusable helper functions to:

- calculate metrics
- plot confusion matrices
- store model comparison results

In [ ]:
# ============================================================
# Helper functions
# ============================================================

def evaluate_predictions(y_true, y_pred, model_name):
    """Compute core multi-class classification metrics."""
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision_Macro": precision,
        "Recall_Macro": recall,
        "F1_Macro": f1
    }

def plot_confusion_matrix_from_labels(y_true, y_pred, class_names, title):
    """Plot confusion matrix with labels."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title)
    plt.show()

# Model 1 — Logistic Regression with TF-IDF

## Why This Model?

This is the simplest and most interpretable text classification baseline.

Pipeline:

1. Convert text into TF-IDF vectors  
2. Train multinomial logistic regression  
3. Predict the stocking priority class  

This provides a strong classical baseline against which deep learning models can be compared.

In [ ]:
# ============================================================
# Logistic Regression + TF-IDF baseline
# ============================================================

logreg_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=1000,
        ngram_range=(1, 2),
        min_df=2
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        random_state=SEED
    ))
])

# Train model
logreg_pipeline.fit(X_train, y_train)

# Validation predictions
val_pred_logreg = logreg_pipeline.predict(X_val)

# Test predictions
test_pred_logreg = logreg_pipeline.predict(X_test)

# Evaluation
logreg_val_metrics = evaluate_predictions(y_val, val_pred_logreg, "Logistic Regression (Validation)")
logreg_test_metrics = evaluate_predictions(y_test, test_pred_logreg, "Logistic Regression (Test)")

print("Validation Metrics:")
print(logreg_val_metrics)

print("\nTest Classification Report:")
print(classification_report(
    y_test,
    test_pred_logreg,
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
# Confusion matrix for Logistic Regression
plot_confusion_matrix_from_labels(
    y_test,
    test_pred_logreg,
    class_names=label_encoder.classes_,
    title="Logistic Regression - Test Confusion Matrix"
)

## Observation

Logistic Regression provides a transparent baseline and is useful for checking whether the business text fields contain enough signal to separate:

- low-priority
- medium-priority
- high-priority

stocking outcomes.

# Model 2 — Custom Deep Learning Text Classifier

## Why a Custom Model?

A custom neural network allows us to learn distributed representations directly from the text without relying purely on TF-IDF counts.

### Architecture Used

- TextVectorization
- Embedding layer
- Bidirectional LSTM
- Dense hidden layer
- Softmax output

This architecture is designed to capture short business-text patterns and their combinations.

In [ ]:
# ============================================================
# Prepare TensorFlow text pipeline
# ============================================================

# Convert training/validation/test data into numpy arrays
X_train_text = np.array(X_train)
X_val_text = np.array(X_val)
X_test_text = np.array(X_test)

y_train_arr = np.array(y_train)
y_val_arr = np.array(y_val)
y_test_arr = np.array(y_test)

# Text vectorization layer
max_tokens = 3000
sequence_length = 30

text_vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Adapt on training text only
text_vectorizer.adapt(X_train_text)

In [ ]:
# ============================================================
# Build custom deep learning architecture
# ============================================================

embedding_dim = 64
num_classes = len(label_encoder.classes_)

custom_model = keras.Sequential([
    layers.Input(shape=(1,), dtype=tf.string),
    text_vectorizer,
    layers.Embedding(input_dim=max_tokens, output_dim=embedding_dim),
    layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

custom_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

custom_model.summary()

In [ ]:
# ============================================================
# Train custom deep learning model
# ============================================================

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = custom_model.fit(
    X_train_text,
    y_train_arr,
    validation_data=(X_val_text, y_val_arr),
    epochs=8,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# ============================================================
# Evaluate custom deep learning model
# ============================================================

# Predict probabilities, then convert to class labels
custom_test_probs = custom_model.predict(X_test_text)
custom_test_pred = np.argmax(custom_test_probs, axis=1)

custom_test_metrics = evaluate_predictions(
    y_test_arr,
    custom_test_pred,
    "Custom Deep Learning Model"
)

print("Custom Deep Learning Model - Test Classification Report")
print(classification_report(
    y_test_arr,
    custom_test_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
# Confusion matrix for custom model
plot_confusion_matrix_from_labels(
    y_test_arr,
    custom_test_pred,
    class_names=label_encoder.classes_,
    title="Custom Deep Learning Model - Test Confusion Matrix"
)

In [ ]:
# Plot training history
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(10, 4))
plt.plot(history_df["loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Custom Model Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Custom Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Observation

The custom deep learning model can learn feature interactions that are harder for linear TF-IDF models to capture.

This is especially useful when short product/category phrases combine in patterns that indicate stronger or weaker stocking priority.

# Model 3 — Hugging Face Pre-trained Transformer

## Why a Pre-trained Model?

Pre-trained transformer models already contain strong language representations learned from large corpora.

In this notebook, we fine-tune a transformer for our business text classification problem.

### Suggested Model

We use:

`distilbert-base-uncased`

This model is a good balance between:
- performance
- training speed
- memory usage

> If Colab GPU memory is limited, you may replace it with a smaller model such as:
> `prajjwal1/bert-tiny`

In [ ]:
# ============================================================
# Prepare Hugging Face datasets
# ============================================================

# Create DataFrames for each split
train_df = pd.DataFrame({"text": X_train.tolist(), "label": y_train.tolist()})
val_df = pd.DataFrame({"text": X_val.tolist(), "label": y_val.tolist()})
test_df = pd.DataFrame({"text": X_test.tolist(), "label": y_test.tolist()})

# Convert to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

In [ ]:
# ============================================================
# Tokenizer and tokenization function
# ============================================================

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=64
    )

tokenized_train = train_dataset.map(tokenize_batch, batched=True)
tokenized_val = val_dataset.map(tokenize_batch, batched=True)
tokenized_test = test_dataset.map(tokenize_batch, batched=True)

In [ ]:
# ============================================================
# Load pre-trained classification model
# ============================================================

hf_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_classes
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# ============================================================
# Metrics function for Hugging Face Trainer
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [ ]:
# ============================================================
# Training arguments
# ============================================================

training_args = TrainingArguments(
    output_dir="hf_text_classifier_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none",
    seed=SEED
)

In [ ]:
# ============================================================
# Train the Hugging Face model
# ============================================================

trainer = Trainer(
    model=hf_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# ============================================================
# Evaluate the Hugging Face model on the test set
# ============================================================

hf_test_output = trainer.predict(tokenized_test)
hf_test_pred = np.argmax(hf_test_output.predictions, axis=1)

hf_test_metrics = evaluate_predictions(
    y_test.to_numpy(),
    hf_test_pred,
    "Hugging Face Transformer"
)

print("Hugging Face Transformer - Test Classification Report")
print(classification_report(
    y_test.to_numpy(),
    hf_test_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
# Confusion matrix for Hugging Face transformer
plot_confusion_matrix_from_labels(
    y_test.to_numpy(),
    hf_test_pred,
    class_names=label_encoder.classes_,
    title="Hugging Face Transformer - Test Confusion Matrix"
)

## Observation

The transformer model benefits from pre-trained contextual representations. Even though the input text is business-oriented and not full natural language, the pre-trained model may still capture useful relationships among product, brand, promotion, and festival descriptors.

# Compare All Three Models

We now compare:

1. Logistic Regression + TF-IDF  
2. Custom Deep Learning Model  
3. Hugging Face Transformer  

The comparison is based on:

- Accuracy
- Macro Precision
- Macro Recall
- Macro F1-score

In [ ]:
# ============================================================
# Build comparison table
# ============================================================

comparison_results = pd.DataFrame([
    {
        "Model": "Logistic Regression + TF-IDF",
        "Accuracy": logreg_test_metrics["Accuracy"],
        "Precision_Macro": logreg_test_metrics["Precision_Macro"],
        "Recall_Macro": logreg_test_metrics["Recall_Macro"],
        "F1_Macro": logreg_test_metrics["F1_Macro"]
    },
    {
        "Model": "Custom Deep Learning Model",
        "Accuracy": custom_test_metrics["Accuracy"],
        "Precision_Macro": custom_test_metrics["Precision_Macro"],
        "Recall_Macro": custom_test_metrics["Recall_Macro"],
        "F1_Macro": custom_test_metrics["F1_Macro"]
    },
    {
        "Model": "Hugging Face Transformer",
        "Accuracy": hf_test_metrics["Accuracy"],
        "Precision_Macro": hf_test_metrics["Precision_Macro"],
        "Recall_Macro": hf_test_metrics["Recall_Macro"],
        "F1_Macro": hf_test_metrics["F1_Macro"]
    }
])

comparison_results.sort_values(by="F1_Macro", ascending=False).reset_index(drop=True)

In [ ]:
# ============================================================
# Visual comparison
# ============================================================

metrics_to_plot = ["Accuracy", "F1_Macro"]

for metric in metrics_to_plot:
    plt.figure(figsize=(9, 4))
    sns.barplot(data=comparison_results, x="Model", y=metric)
    plt.title(f"Model Comparison - {metric}")
    plt.xticks(rotation=15)
    plt.show()

# Business Interpretation

This notebook reframes the dataset as a **stocking decision classification problem**.

Instead of asking only:

> "How many units will sell?"

we ask:

- Which product records belong to low, medium, or high stocking priority?
- Which category/brand/promotion combinations are associated with stronger profit density?
- Which textual business descriptors help identify more attractive stocking opportunities?

This perspective is especially useful in real retail strategy where shelf space, product mix, and profit efficiency matter.

# Final Conclusion

In this notebook, the updated retail dataset was analyzed from a **product category and profitability classification perspective**.

## What was done

- A business-driven multi-class target, `Stocking_Priority_Class`, was created using profit density.
- Text-like business descriptors were combined into a single input representation.
- Three model families were compared:
  1. Logistic Regression + TF-IDF
  2. Custom Deep Learning Classifier
  3. Hugging Face Pre-trained Transformer

## Why this matters

This analysis helps move from pure demand estimation toward a more decision-oriented question:

> Which product-category records should be prioritized for stocking based on profitability and selling efficiency?

## Expected Learning Outcome

- Logistic Regression provides a strong interpretable baseline
- The custom deep model learns nonlinear text patterns
- The transformer model tests whether pre-trained language knowledge improves classification on retail business text

# Suggested Next Extensions

1. Replace the combined text field with **column-aware text templates**  
   Example:

   `Category: Beauty | Product: L'oreal Makeup 1002 | Brand Tier: Premium | Promotion: Flash Sale`

2. Add structured numeric variables to a hybrid model  
   Example:
   - current price
   - shelf capacity
   - lead time
   - marketing spend

3. Compare:
   - text-only classification
   - numeric-only classification
   - hybrid text + numeric classification

4. Analyze which categories appear most often in the **High** stocking priority class.